# Task 13: Temporal Convolutional Network with dilated causal convolutions

In [1]:
import torch
import torch.nn as nn


In [2]:
class CausalConv1d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation):
        super().__init__()
        self.pad = (kernel_size-1)*dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size, dilation=dilation, padding=self.pad)

    def forward(self, x):
        out = self.conv(x)
        return out[:, :, :-self.pad] if self.pad != 0 else out

class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation):
        super().__init__()
        self.conv1 = CausalConv1d(in_ch, out_ch, kernel_size, dilation)
        self.conv2 = CausalConv1d(out_ch, out_ch, kernel_size, dilation)
        self.relu = nn.ReLU()
        self.downsample = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None

    def forward(self, x):
        out = self.relu(self.conv1(x))
        out = self.relu(self.conv2(out))
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)


In [3]:
class TCN(nn.Module):
    def __init__(self, in_ch, channels, kernel_size=3):
        super().__init__()
        layers = []
        for i, ch in enumerate(channels):
            dilation = 2**i
            layers.append(TCNBlock(in_ch if i==0 else channels[i-1], ch, kernel_size, dilation))
        self.net = nn.Sequential(*layers)
        self.head = nn.Conv1d(channels[-1], 1, 1)

    def forward(self, x):
        return self.head(self.net(x))


In [4]:
model = TCN(in_ch=1, channels=[16,16,16,16])
x = torch.randn(4, 1, 100)
out = model(x)
print(out.shape)


torch.Size([4, 1, 100])


In [5]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
target = torch.sin(torch.linspace(0, 10, 100)).view(1,1,-1).repeat(4,1,1)

for epoch in range(50):
    opt.zero_grad()
    pred = model(x)
    loss = torch.nn.functional.mse_loss(pred, target)
    loss.backward()
    opt.step()

print("final loss", loss.item())


final loss 0.21442510187625885
